In [12]:
from sentence_transformers import util
from sentence_transformers import SentenceTransformer
from skill_extractor import extract_skills
import warnings
warnings.filterwarnings("ignore")

In [13]:
# --- 1. Simulate a Job Description from the Database ---
# (In the real app, this comes from the Himalayas API or an Employer post)
job_raw_text = """
We are hiring a Senior Backend Developer. 
You must have strong experience with Python and Django. 
Familiarity with REST APIs and PostgreSQL is required. 
Knowledge of Docker and AWS is a big plus.
"""

# Extract skills for the Job (using the same logic as the resume)
job_skills = list(set([skill.lower().strip() for skill in extract_skills(job_raw_text)]))

job_data = {
    "title": "Senior Backend Developer",
    "raw_text": job_raw_text,
    "required_skills": job_skills
}

In [14]:
# --- 2. The Matching Function ---
def match_resume_to_job(resume_data, job_data, embedder):
    print(f"\nComparing Resume to: {job_data['title']}")
    
    # A. Semantic Score (Cosine Similarity)
    # Note: We assume resume_data['embedding'] is already generated
    # We generate the job embedding right here for the demo
    job_embedding = embedder.encode(job_data['raw_text'])
    
    # Calculate similarity
    cosine_score = util.cos_sim(resume_data['embedding'], job_embedding).item()
    
    # B. Skill Gap Analysis (Set Intersection)
    resume_skills = set(resume_data['extracted_skills'])
    required_skills = set(job_data['required_skills'])
    
    matched_skills = list(resume_skills.intersection(required_skills))
    missing_skills = list(required_skills.difference(resume_skills))
    
    # C. Output the Results
    print("-" * 30)
    print(f"🎯 Semantic Match Score: {cosine_score * 100:.2f}%")
    print(f"✅ Matched Skills: {matched_skills}")
    print(f"❌ Missing Skills: {missing_skills}")
    print("-" * 30)
    
    # D. Return the payload (This is exactly what goes into your 'matches' table in the ERD)
    return {
        "score": cosine_score,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills
    }

In [15]:
embedder = SentenceTransformer('all-MiniLm-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3533.04it/s]


In [18]:
# --- 3. Run the Test ---
# (Paste the result dictionary you got from processing your test_resume.pdf here)
# For testing purposes, I am using a mock resume payload:
mock_resume_data = {
    "extracted_skills": ['postgresql', 'software development', 'git & github', 'python', 'troubleshooting', 'github', 'rest api', 'git', 'javascript', 'django', 'django rest framework', 'web development', 'rest apis', 'react'],
    "embedding": [-0.08880642056465149, -0.016362277790904045, -0.004568162374198437, 0.03641720861196518, -0.053067661821842194, -0.08511140942573547, -0.06712913513183594, 0.03571194037795067, -0.08776833117008209, 0.004886539652943611, -0.017636891454458237, -0.06869921088218689, 0.02201201766729355, 0.018891727551817894, 0.04681013897061348, 0.05633913725614548, 0.07063111662864685, -0.04801589250564575, 0.03729558736085892, -0.09451086074113846, -0.041655197739601135, 0.021740397438406944, 0.03423832356929779, -0.05449342727661133, 0.02332535944879055, -0.005373755935579538, -0.025588510558009148, -0.03188792243599892, -0.06694750487804413, -0.08256053179502487, -0.019321991130709648, 0.003619285300374031, -0.011358022689819336, 0.10178960114717484, 0.028336480259895325, 0.09504617005586624, 0.008752281777560711, -0.02260727807879448, 0.03932129964232445, -0.03197446092963219, -0.07204733043909073, -0.0319385901093483, -0.06515184044837952, -0.03317935764789581, 0.004478569142520428, -0.09853851795196533, -0.03507968783378601, -0.08589683473110199, 0.05647268518805504, -0.005510415881872177, -0.06288062781095505, -0.10396626591682434, 0.0394882932305336, -0.03441108018159866, -0.0701388344168663, -0.0013528403360396624, 0.015325119718909264, -0.028601327911019325, -0.0008682106272317469, -0.021239304915070534, 0.01919773779809475, -0.01093703880906105, 0.006589126773178577, 0.034922052174806595, -0.041669297963380814, -0.06787329912185669, -0.05241645127534866, 0.034503303468227386, 0.06618708372116089, -0.08091653883457184, -0.021420219913125038, 0.0006076859426684678, -0.11075452715158463, 0.022038808092474937, 0.03601105883717537, -0.04339175671339035, 0.02615281008183956, 0.0052710589952766895, 0.04295838251709938, -0.011178391985595226, 0.0028528065886348486, 0.005642222240567207, -0.08393273502588272, 0.10052576661109924, -0.0982770174741745, -0.011038935743272305, 0.06470061093568802, 0.10985660552978516, -0.006174256559461355, -0.028806721791625023, 0.06094484403729439, -0.03364447504281998, 0.05968869477510452, -0.011630126275122166, 0.012792134657502174, -0.03136999532580376, 0.09494500607252121, -0.010030264966189861, -0.08482612669467926, 0.029319951310753822, -0.0014554569497704506, -0.06038525328040123, 0.05544527247548103, -0.002081963000819087, -0.07448876649141312, -0.039043162018060684, 0.01019153743982315, 0.005963644478470087, 0.04783517122268677, 0.012385581620037556, -0.06029364466667175, 0.041411079466342926, -0.12418431043624878, -0.00024213604046963155, 0.04097737744450569, 0.0642094686627388, -0.05243130773305893, 0.027601107954978943, 0.010701661929488182, 0.0699314996600151, 0.06471346318721771, 0.10433731228113174, -0.010756881907582283, -0.044646523892879486, 0.029848501086235046, -0.02646838128566742, -0.010631582699716091, 7.752244919034857e-33, 0.09912858158349991, 0.07317987084388733, 0.041510291397571564, 0.038644369691610336, 0.07885362952947617, -0.009471194818615913, 0.03691988065838814, 0.08513090759515762, -0.03792025148868561, -0.054135583341121674, -0.0626007616519928, 0.10660286247730255, 0.011416580528020859, 0.05100005865097046, 0.019508657976984978, 0.014595312997698784, -0.03990425914525986, 0.031802091747522354, -0.009016434662044048, 0.07735568284988403, -0.00852642860263586, -0.10213232785463333, -0.030978666618466377, 0.07489067316055298, 0.05602628365159035, 0.012258071452379227, 0.04183309152722359, 0.04423413798213005, 0.019729891791939735, 0.03122236765921116, -0.060308102518320084, -0.02800101600587368, -0.06592739373445511, -0.03201527148485184, -0.020614180713891983, 0.03962650150060654, -0.014637497253715992, -0.08845554292201996, -0.008499856106936932, -0.04181484133005142, -0.045609250664711, 0.06338247656822205, -0.01622248999774456, -0.04064234346151352, -0.03242701664566994, -0.024462049826979637, 0.06734674423933029, 0.058307286351919174, 0.066764235496521, 0.08155098557472229, -0.06908441334962845, -0.02329276129603386, 0.03310346230864525, -0.004102489911019802, -0.038677603006362915, 0.006416993215680122, 0.052908312529325485, 0.004970917012542486, -0.04123074933886528, 0.012940993532538414, 0.052788883447647095, -0.04006186127662659, -0.04435763508081436, 0.04956210404634476, -0.027158834040164948, -0.05718561261892319, 0.04474181309342384, 0.010820521973073483, 0.18304555118083954, 0.024556314572691917, -0.048282429575920105, 0.030621273443102837, 0.06931035965681076, 0.01951364241540432, -0.0726933628320694, -0.03872670605778694, -0.03978937864303589, -0.008778728544712067, 0.023253943771123886, 0.04585909843444824, 0.030648240819573402, 0.02530515380203724, -0.03182166814804077, -0.06212325021624565, -0.0096852146089077, 0.03336568921804428, 0.04270775243639946, 0.016039252281188965, -0.02839653566479683, 0.07145678251981735, -0.029768936336040497, -0.0022704042494297028, 0.03327932953834534, 0.06177309527993202, -0.005983490962535143, -8.510129473364172e-33, 0.03083021193742752, -0.08491864055395126, 0.0021428295876830816, -0.023558123037219048, 0.10873346030712128, -0.030896814540028572, 0.11673644185066223, 0.08632460981607437, 0.0035701002925634384, -0.0005307852406986058, 0.0014005550183355808, 0.001159134553745389, 0.024667009711265564, -0.0016574850305914879, 0.00026584023726172745, 0.01760241389274597, -0.10874227434396744, -0.03304020315408707, 0.01112201064825058, -0.0035264978650957346, -0.015426039695739746, 0.09700734913349152, -0.0597289502620697, -0.02688058279454708, 0.0203646719455719, 0.009192897006869316, -0.04985653981566429, -0.04255920648574829, -0.04751073941588402, -0.030958877876400948, 0.05177050828933716, -0.017723701894283295, -0.034915465861558914, 0.004437598865479231, -0.05746885761618614, -0.02217266894876957, -0.02767065167427063, 0.01360953226685524, 0.031019793823361397, 0.0635615810751915, 0.10870463401079178, 0.011379286646842957, 0.02891371212899685, -0.05320471152663231, -0.004151520319283009, -0.0339154414832592, -0.014745625667273998, 0.0038654599338769913, -0.023411057889461517, -0.09740070253610611, -0.043167922645807266, -0.032541438937187195, -0.026966257020831108, -0.043836262077093124, 0.029866870492696762, -0.01706135831773281, 0.03640446811914444, -0.028654208406805992, -0.07787167280912399, -0.04304952919483185, -0.017196808010339737, 0.013557801954448223, 0.1062176525592804, 0.10669902712106705, -0.0606868714094162, -0.07310231775045395, -0.01764129288494587, -0.04865669831633568, -0.11198645085096359, -0.07377184182405472, -0.03224019706249237, -0.05390544980764389, 0.026676146313548088, -0.0381234735250473, -0.02037067897617817, 7.335522241191939e-05, -0.1144905835390091, -0.02425316534936428, -0.04136990010738373, -0.012077368795871735, -0.07228744029998779, -0.02177102491259575, 0.013468246906995773, 0.08212760835886002, 0.008062264882028103, -0.0007357753347605467, 0.0077004143968224525, 0.026219576597213745, 0.01755373738706112, -0.029265929013490677, -0.06201549619436264, -0.025712579488754272, -0.030794166028499603, 0.0446724072098732, -0.0022030496038496494, -5.8141406356071457e-08, 0.008590208366513252, 0.03245845437049866, -0.02387338876724243, 0.06711462140083313, -0.03070756420493126, 0.055448222905397415, -0.0013154103653505445, 0.048758819699287415, 0.07142242044210434, 0.034655116498470306, -0.03230639919638634, -0.05681481584906578, -0.06842872500419617, 0.0497044175863266, 0.09163164347410202, 0.021720677614212036, 0.00773664191365242, 0.0975038930773735, -0.017258504405617714, -0.0663129910826683, 0.023556197062134743, -0.0144581850618124, 0.017446745187044144, 0.01080300286412239, 0.017799926921725273, 0.04925289750099182, -0.051104869693517685, 0.06796615570783615, 0.006152510643005371, -0.05330069735646248, -0.0036122326273471117, -0.005524660926312208, 0.034863922744989395, -0.09433648735284805, 0.08532059192657471, -0.05519264191389084, 0.034681301563978195, -0.04079184681177139, 0.006505127530544996, 0.09164832532405853, 0.011767315678298473, -0.017946757376194, 0.012645485810935497, 0.008470851927995682, 0.07992912828922272, 0.019296664744615555, -0.06713226437568665, -0.0605885349214077, 0.04334079846739769, -0.04196000099182129, -0.014558264054358006, -0.0954534187912941, 0.00719778286293149, 0.006777509115636349, 0.05420501157641411, 0.11326709389686584, -0.028384702280163765, -0.050962500274181366, 0.009596651419997215, 0.050104860216379166, 0.14664581418037415, -0.033042844384908676, -0.049832578748464584, 0.04745631292462349]
}

# Run the matching
match_result = match_resume_to_job(mock_resume_data, job_data, embedder)


Comparing Resume to: Senior Backend Developer
------------------------------
🎯 Semantic Match Score: 65.37%
✅ Matched Skills: ['rest apis', 'django', 'python', 'postgresql']
❌ Missing Skills: ['docker', 'aws']
------------------------------
